New Notebook Created by Jupyter MCP Server

In [1]:
from pathlib import Path
import sys, os
REPO = Path('/home/nahisaho/GitHub/jupytermind')
sys.path.insert(0, str(REPO / 'src'))
from ai_data_scientist import project_manager, lifecycle, language_router
handle = project_manager.resolve_project('kaggle-v020-kaggle-exp-49-water-quality', projects_root=REPO / 'benchmark/projects')
project_manager.ensure_notebook(handle)
print('cwd:', Path.cwd(), '\nnotebook:', handle.notebook_path)
print('language:', language_router.detect_language('Kaggleの水質データを分析して日本語で報告してください'))

cwd: /home/nahisaho/GitHub/jupytermind 
notebook: /home/nahisaho/GitHub/jupytermind/benchmark/projects/kaggle-v020-kaggle-exp-49-water-quality/notebooks/kaggle-v020-kaggle-exp-49-water-quality.ipynb
language: ja


In [2]:
import nbformat
bootstrap_source = r'''from pathlib import Path
import sys, os, json, hashlib, io, contextlib, traceback, re
from datetime import datetime, timezone
from dataclasses import asdict
REPO = Path('/home/nahisaho/GitHub/jupytermind')
sys.path.insert(0, str(REPO / 'src'))
from ai_data_scientist import project_manager, lifecycle, language_router
RUN_ID = 'v020-exp-49'
handle = project_manager.resolve_project('kaggle-v020-kaggle-exp-49-water-quality', projects_root=REPO / 'benchmark/projects')
project_manager.ensure_notebook(handle)
data_dir = project_manager.ensure_data_dir(handle)
lifecycle.register_run(RUN_ID, handle.notebook_path)
print('言語:', language_router.detect_language('水質と飲用可否の関係を日本語で分析してください'))
phase_log = []
@contextlib.contextmanager
def phase(name, writing=False):
    if lifecycle.is_cancel_requested(RUN_ID):
        lifecycle.mark_failed(RUN_ID)
        raise RuntimeError('キャンセル要求を検出')
    start = lifecycle.mark_write_start if writing else lifecycle.mark_execution_start
    end = lifecycle.mark_write_end if writing else lifecycle.mark_execution_end
    start(RUN_ID)
    phase_log.append({'phase': name, 'event': 'start', 'utc': datetime.now(timezone.utc).isoformat()})
    try:
        yield
    except BaseException:
        lifecycle.mark_failed(RUN_ID)
        phase_log.append({'phase': name, 'event': 'failed'})
        raise
    finally:
        end(RUN_ID)
        phase_log.append({'phase': name, 'event': 'end', 'utc': datetime.now(timezone.utc).isoformat()})
def write_cells(cells):
    with phase('Notebook書込み', writing=True):
        project_manager.enqueue_write(handle, lambda nb: nb.cells.extend(cells))
print('保存先:', handle.notebook_path)
print(asdict(lifecycle.get_run_status(RUN_ID)))
'''
project_manager.enqueue_write(handle, lambda nb: nb.cells.extend([nbformat.v4.new_markdown_cell('# 水質と飲用可否：規制の条件比較とデータ駆動の関連\n\nrun_id: `v020-exp-49`。Kaggleの公開データを探索・取得し、単位とラベルの不確実性を明示する。法的な飲用判定や因果推論は行わない。分析/API実行はJupyter MCPのみ。'), nbformat.v4.new_code_cell(bootstrap_source)]))
print('対象Notebookを作成しました')

対象Notebookを作成しました


In [4]:
acquisition_source = r'''import pandas as pd
import numpy as np
import requests, zipfile, inspect
from IPython.display import display, Markdown
acquisition_log = []
slug = None
filename = None
with phase('Kaggle認証・検索・取得'):
    try:
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            from kaggle.api.kaggle_api_extended import KaggleApi
            api = KaggleApi()
            api.authenticate()
            candidates = api.dataset_list(search='water potability')
        refs = [str(d.ref) for d in candidates]
        print('Kaggle検索 water potability:', refs)
        if 'adityakadiwal/water-potability' in refs:
            slug = 'adityakadiwal/water-potability'
        elif 'sulianova/water-potability' in refs:
            slug = 'sulianova/water-potability'
        else:
            raise RuntimeError('対応する飲用可否データのslugを検索結果で確認できません')
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            files_response = api.dataset_list_files(slug)
        file_names = [f.name for f in files_response.files]
        print('slug:', slug, '公開ファイル:', file_names)
        if 'water_potability.csv' not in file_names:
            raise RuntimeError('対応CSVがファイル一覧にありません')
        filename = 'water_potability.csv'
        raw_path = data_dir / filename
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            api.dataset_download_file(slug, filename, path=str(data_dir), force=True, quiet=True)
        if not raw_path.exists():
            archives = list(data_dir.glob('water_potability.csv*.zip'))
            if len(archives) != 1:
                raise RuntimeError('取得CSVまたは対応zipを確認できません')
            with zipfile.ZipFile(archives[0]) as archive:
                if filename not in archive.namelist():
                    raise RuntimeError('zip内に対応CSVがありません')
                raw_path.write_bytes(archive.read(filename))
        acquisition_log.append({'stage': 'Kaggle API', 'status': '取得成功'})
        source_kind = 'Kaggle API'
    except (Exception, SystemExit) as exc:
        acquisition_log.append({'stage': 'Kaggle API', 'status': '取得失敗', 'exception_type': type(exc).__name__, 'http_status': getattr(getattr(exc, 'response', None), 'status_code', None)})
        whitepaper = Path('/home/nahisaho/kaggle/experiments/white-paper.md')
        if not whitepaper.exists():
            print(acquisition_log)
            raise RuntimeError('取得失敗：指定white-paper.mdが存在せずフォールバック不可') from None
        text = whitepaper.read_text(encoding='utf-8')
        relevant = [line for line in text.splitlines() if 'water_potability' in line or 'water-potability' in line]
        public_urls = [u.rstrip(').,') for line in relevant for u in re.findall(r'https?://[^\s<>"`]+', line) if '.csv' in u]
        if len(set(public_urls)) != 1:
            print(acquisition_log)
            print('white-paper該当行:', relevant)
            raise RuntimeError('取得失敗：同一データの一意な公開CSVをwhite-paperで確認できません') from None
        fallback_url = public_urls[0]
        print('white-paperに明記された対応CSV:', fallback_url)
        response = requests.get(fallback_url, timeout=60)
        response.raise_for_status()
        filename = 'water_potability.csv'
        raw_path = data_dir / filename
        raw_path.write_bytes(response.content)
        source_kind = 'white-paper明記の同一CSV'
        acquisition_log.append({'stage': 'white-paper fallback', 'status': '取得成功', 'url': fallback_url})
    provenance = {'owner_dataset_slug': slug, 'filename': filename, 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(), 'sha256': hashlib.sha256(raw_path.read_bytes()).hexdigest(), 'source_kind': source_kind, 'run_id': RUN_ID}
    df = pd.read_csv(raw_path)
    assert set(df.columns) == {'ph','Hardness','Solids','Chloramines','Sulfate','Conductivity','Organic_carbon','Trihalomethanes','Turbidity','Potability'}
    print(json.dumps(provenance, ensure_ascii=False, indent=2))
    print('API取得ログ（例外本文・認証情報は記録しない）:', acquisition_log)
    display(df.head())
    print('形状:', df.shape)
    with phase('取得manifest書込み', writing=True):
        (data_dir / 'provenance.json').write_text(json.dumps(provenance, ensure_ascii=False, indent=2), encoding='utf-8')
'''
write_cells([nbformat.v4.new_markdown_cell('## 1. データ取得と来歴\n\nSDK認証情報を出力しない。検索結果とファイル一覧で対応データを確認する。取得失敗時のみ指定稿の同一CSVを確認する。'), nbformat.v4.new_code_cell(acquisition_source)])
print('取得セルを保存しました')

取得セルを保存しました


In [6]:
metadata_source = r'''from bs4 import BeautifulSoup
with phase('辞書・規制一次資料の確認'):
    dictionary_text = ''
    metadata_status = 'unknown'
    if source_kind == 'Kaggle API':
        try:
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                api.dataset_metadata(slug, path=str(data_dir))
            metadata_files = list(data_dir.glob('*metadata*.json'))
            for metadata_file in metadata_files:
                metadata = json.loads(metadata_file.read_text(encoding='utf-8'))
                description = metadata.get('description', '')
                if description:
                    dictionary_text = description
                    metadata_status = 'verified'
                    print('Kaggle説明:', description)
                    print('license:', metadata.get('licenses'))
        except Exception as exc:
            print('データ辞書API失敗:', type(exc).__name__, '単位・定義を未確認として扱う')
    reference_urls = {
        'EPA_primary': 'https://www.epa.gov/ground-water-and-drinking-water/national-primary-drinking-water-regulations',
        'EPA_secondary': 'https://www.epa.gov/sdwa/secondary-drinking-water-standards-guidance-nuisance-chemicals'
    }
    references = {}
    for name, url in reference_urls.items():
        try:
            response = requests.get(url, timeout=45)
            response.raise_for_status()
            page_text = BeautifulSoup(response.text, 'html.parser').get_text(' ', strip=True)
            references[name] = {'url': url, 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(), 'sha256': hashlib.sha256(response.content).hexdigest(), 'text': page_text, 'status': 'verified'}
            (data_dir / (name + '.html')).write_bytes(response.content)
            for keyword in (['Chloramines', 'Total Trihalomethanes', 'Turbidity'] if name == 'EPA_primary' else ['pH', 'Sulfate', 'Total Dissolved Solids']):
                matches = list(re.finditer(re.escape(keyword), page_text, flags=re.I))
                excerpts = [page_text[max(0,m.start()-100):m.end()+450] for m in matches[:3]]
                print(name, keyword, excerpts)
        except requests.RequestException as exc:
            references[name] = {'url': url, 'status': 'unknown', 'exception_type': type(exc).__name__}
            print('規制資料取得失敗:', name, type(exc).__name__)
    print('単位の説明取得:', metadata_status)
'''
write_cells([nbformat.v4.new_markdown_cell('## 2. データ辞書と規制資料\n\nKaggle掲載説明の確認と測定の真正性は別である。比較枠は米国EPAの一次・二次基準とし、日本法・WHO基準との混用を避ける。'), nbformat.v4.new_code_cell(metadata_source)])

In [8]:
definition_source = r'''from ai_data_scientist import data_definition, analysis_assumptions, data_quality, eda, cleaning, stats_analysis
with phase('データ定義・品質・欠損確認'):
    dictionary_url = 'https://www.kaggle.com/api/v1/datasets/view/' + (slug or 'adityakadiwal/water-potability')
    try:
        response = requests.get(dictionary_url, timeout=45)
        response.raise_for_status()
        public_metadata = response.json()
        dictionary_text = public_metadata.get('description', '')
        if dictionary_text:
            metadata_status = 'verified'
            (data_dir / 'public-dataset-description.txt').write_text(dictionary_text, encoding='utf-8')
            print('公開データ辞書:', dictionary_text)
        else:
            print('公開メタデータにdescriptionなし。キー:', sorted(public_metadata.keys()))
    except (requests.RequestException, ValueError) as exc:
        print('公開辞書取得不可:', type(exc).__name__, '単位は未確認/推定として記録する')
    units = {'ph':'無次元', 'Hardness':'mg/L（CaCO3換算は未確認）', 'Solids':'ppm', 'Chloramines':'ppm', 'Sulfate':'mg/L', 'Conductivity':'µS/cm', 'Organic_carbon':'ppm', 'Trihalomethanes':'µg/L', 'Turbidity':'NTU', 'Potability':'0/1'}
    definitions = {'ph':'水素イオン活量の指標', 'Hardness':'硬度', 'Solids':'総溶解固形物 TDS', 'Chloramines':'クロラミン', 'Sulfate':'硫酸塩', 'Conductivity':'電気伝導率', 'Organic_carbon':'総有機炭素', 'Trihalomethanes':'トリハロメタン', 'Turbidity':'濁度', 'Potability':'0=飲用不可、1=飲用可（掲載説明のラベル）'}
    def fv(value, status, source=None):
        return data_definition.FieldValue(value, status, source)
    unit_status = 'verified' if dictionary_text else 'inferred'
    definition_manifest = data_definition.build_manifest(
        source={k:fv(v,'verified','Kaggle API/取得ファイル') for k,v in provenance.items()},
        dataset_scope={'site':fv(None,'unknown'), 'measurement_date':fv(None,'unknown'), 'sampling_frame':fv(None,'unknown'), 'label_generation_rule':fv(None,'unknown'), 'measurement_method':fv(None,'unknown'), 'authenticity_or_synthetic':fv(None,'unknown')},
        variables={c:{'unit':fv(units[c], unit_status, dictionary_url), 'definition':fv(definitions[c],unit_status,dictionary_url), 'measurement_comparability':fv(None,'unknown')} for c in df.columns},
        transformations=('元CSVを保持。重複の確認後に除去、主解析は列別の観測済み値。法的判定は行わない。',))
    print('データ定義manifest:', json.dumps(asdict(definition_manifest), ensure_ascii=False, indent=2))
    print('未解決項目:', [(p,v.status) for p,v in definition_manifest.unresolved_fields()])
    print('推定項目:', [(c,k,v.value) for c,fields in definition_manifest.variables.items() for k,v in fields.items() if v.status == 'inferred'])
    dedup = cleaning.clean_dataset(df, operation='drop_duplicates')
    analysis_df = dedup.dataframe.copy()
    features = analysis_df.columns.drop('Potability').tolist()
    quality_report = eda.explore(analysis_df)
    print('重複除去:', {'rows_before':dedup.rows_before,'rows_after':dedup.rows_after,'rows_removed':dedup.rows_removed})
    display(pd.DataFrame(quality_report.missing_summary).T)
    display(analysis_df.describe().T)
    classes = analysis_df.Potability.value_counts().sort_index().rename('count').to_frame()
    classes['ratio'] = classes['count'] / len(analysis_df)
    display(classes)
    complete_report = cleaning.clean_dataset(analysis_df, operation='drop_na')
    complete_df = complete_report.dataframe
    print('完全ケース:',len(complete_df),'除外:',complete_report.rows_removed)
    display(pd.crosstab(analysis_df.Potability, analysis_df[features].isna().any(axis=1), normalize='index').rename(columns={False:'全指標観測',True:'少なくとも1列欠損'}))
    semantic_schema = {c:{'min':0,'not_null':True} for c in features}
    semantic_schema['ph'] = {'min':0,'max':14,'not_null':True}
    semantic_schema['Potability'] = {'allowed':[0,1],'not_null':True}
    anomaly_records = data_quality.detect_anomalies(analysis_df, schema=semantic_schema)
    print('意味的異常:', [{k:v for k,v in asdict(a).items() if k != 'row_indices'} for a in anomaly_records])
    print('無限値:', int(np.isinf(analysis_df.select_dtypes('number').to_numpy()).sum()))
    assumption_manifest = analysis_assumptions.AnalysisAssumptionManifest(
        analysis_scope={'target':'このCSV内の飲用ラベルとの関連','regulatory':'EPA数値との条件付き比較のみ','external_validity':'保証しない'},
        causal_scope='associational',
        assumptions=(
            analysis_assumptions.Assumption('units-comparable','掲載単位と規制指標の測定定義が同一','assumed',conclusion_critical=True,impact_if_false='基準比較が無効'),
            analysis_assumptions.Assumption('label-valid','Potabilityが監査可能な飲用安全性の正解','assumed',conclusion_critical=True,impact_if_false='予測は掲載ラベルの再現にすぎない'),
            analysis_assumptions.Assumption('sampling-iid','行が独立で同じ母集団から採取された','assumed',conclusion_critical=True,impact_if_false='検定・CVの不確実性が過小評価'),
            analysis_assumptions.Assumption('causal-excluded','観測関連から因果効果を主張しない','verified'),
            analysis_assumptions.Assumption('missing-at-random','欠損が無作為である','assumed',conclusion_critical=True,impact_if_false='完全ケース/補完に偏り')))
    assumption_findings = analysis_assumptions.check_manifest(assumption_manifest)
    print('分析前提manifest:', json.dumps(asdict(assumption_manifest), ensure_ascii=False, indent=2))
    print('前提チェック全所見:', [asdict(f) for f in assumption_findings])
    print('適用不可: 独立の参照データが提供されていないため validate_anomalies / compare_datasets は未実施。時空間情報・採水IDがないため時間/地域評価・因果推論・独立性検証も不可。mcp_gateway はkernel内からCLIのMCP clientを利用できず自己再帰実行を避け、MCP execute_cellを直接利用。')
    with phase('manifest書込み', writing=True):
        (data_dir/'data-definition-manifest.json').write_text(json.dumps(asdict(definition_manifest),ensure_ascii=False,indent=2),encoding='utf-8')
        (data_dir/'analysis-assumptions-manifest.json').write_text(json.dumps(asdict(assumption_manifest),ensure_ascii=False,indent=2),encoding='utf-8')
'''
write_cells([nbformat.v4.new_markdown_cell('## 3. データ定義manifest・分析前提manifest・意味的異常\n\n欠損は測定値に置き換えず、主解析は指標ごとに利用可能な観測値を用いる。範囲外値と基準超過は区別する。'), nbformat.v4.new_code_cell(definition_source)])